#### Confirm chromadb and anthropic are installed correctly
#### New notebook for the RAG/LLM layer. Checking both packages import cleanly
#### before building anything on top of them.

In [1]:
import chromadb
import anthropic

print("chromadb imported successfully")
print("anthropic sdk imported successfully")

client = chromadb.Client()
print("chromadb client created")

chromadb imported successfully
anthropic sdk imported successfully
chromadb client created


In [2]:
import sys
print(sys.executable)

C:\Projects\BA_Airline_Commercial_Analytics\venv\Scripts\python.exe


#### Build the competitor intelligence document corpus
#### Real, current competitor news paraphrased into short summaries -- covering
#### Southwest's Elliott-driven restructuring, Spirit Airlines' complete
#### shutdown in May 2026, and JetBlue's premium-cabin pivot. This is the corpus
#### the RAG assistant will search over.

In [3]:
documents = [
    {"id": "sw_1", "text": "Southwest Airlines came under pressure from activist investor Elliott Management, which built an 11% stake in mid-2024 and forced a board settlement adding six new independent directors in November 2024, ending decades of the founder-led governance style."},
    {"id": "sw_2", "text": "Southwest ended its 50-year 'Bags Fly Free' policy in 2025, introducing checked-bag fees for most fare tiers, and began rolling out assigned seating in place of its long-standing open-boarding model."},
    {"id": "sw_3", "text": "Southwest restructured its fare classes into Basic, Choice, Choice Preferred, and Choice Extra tiers, moving toward the segmented pricing structure long used by legacy carriers like Delta and United."},
    {"id": "sw_4", "text": "Southwest projected roughly $500 million in annual cost savings by 2027 through hiring restraint, scheduling optimization, and fleet modernization, alongside a $2.5 billion share repurchase program approved by its board."},
    {"id": "sw_5", "text": "Analysts and industry commentators including Barclays and J.P. Morgan characterized Southwest's pre-2024 commercial strategy as outdated, citing its slower adoption of ancillary revenue streams compared to peers."},

    {"id": "nk_1", "text": "Spirit Airlines ceased all flight operations permanently at approximately 3am ET on May 2, 2026, after failing to secure a federal bailout, marking the largest US airline shutdown in decades."},
    {"id": "nk_2", "text": "Spirit's collapse followed two bankruptcy filings within less than a year -- November 2024 and August 2025 -- and was ultimately triggered by a spike in jet fuel prices following a Middle East conflict in early 2026."},
    {"id": "nk_3", "text": "Approximately 17,000 Spirit employees and contractors lost their jobs in the shutdown; American, JetBlue, Southwest, and United Airlines absorbed thousands of stranded Spirit passengers in the immediate aftermath."},
    {"id": "nk_4", "text": "As part of its bankruptcy wind-down, Spirit sold preferential gate leases at Chicago O'Hare to American Airlines and United Airlines for a combined roughly $60 million, with proceeds directed toward debtor-in-possession loan repayment."},
    {"id": "nk_5", "text": "Spirit's shutdown effectively removed the primary ultra-low-cost carrier from many US domestic routes, reducing the competitive/price-checking pressure that budget carriers had historically placed on legacy airline fares."},

    {"id": "b6_1", "text": "JetBlue introduced 'BlueFirst Base,' a stripped-down first-class fare that removes perks like seat selection in exchange for a lower price, following similar 'basic premium' moves by United and Delta."},
    {"id": "b6_2", "text": "JetBlue restructured its fare architecture around a two-step choice: passengers first select a cabin experience (Main, EvenMore, BlueFirst, or Mint), then a fare flexibility tier (Base, Standard, or Flex)."},
    {"id": "b6_3", "text": "JetBlue's 'JetForward' turnaround plan emphasizes premium and loyalty revenue growth; transatlantic revenue per available seat mile rose 28% despite reduced capacity, even as core domestic demand softened in early 2025."},
    {"id": "b6_4", "text": "JetBlue opened its first airport lounge, BlueHouse, at JFK, as part of a broader push to compete for premium and corporate travelers typically served by legacy carriers."},
    {"id": "b6_5", "text": "JetBlue cited fare segmentation as a key lever for offsetting fuel cost volatility, alongside Southwest and Delta, both of which reported similar success expanding basic-fare tiers to drive trade-up purchases."},

    {"id": "trend_1", "text": "Across the US airline industry in 2025-2026, multiple carriers moved toward more granular fare segmentation -- unbundling cabins into base, standard, and flexible tiers -- as a shared strategy for capturing revenue from both price-sensitive and premium travelers within the same cabin."},
    {"id": "trend_2", "text": "The shift toward assigned seating and checked-bag fees among historically no-frills carriers reflects broader margin pressure across US airlines following elevated fuel costs and slower post-pandemic demand recovery."},
]

print(f"Corpus built: {len(documents)} documents")
for doc in documents[:3]:
    print(f"\n[{doc['id']}] {doc['text'][:80]}...")

Corpus built: 17 documents

[sw_1] Southwest Airlines came under pressure from activist investor Elliott Management...

[sw_2] Southwest ended its 50-year 'Bags Fly Free' policy in 2025, introducing checked-...

[sw_3] Southwest restructured its fare classes into Basic, Choice, Choice Preferred, an...


#### Embed the corpus into ChromaDB
#### Creates a collection (like a table, but for searchable text) and adds all
#### 17 documents into it. ChromaDB automatically converts each document's text
#### into a vector under the hood - we don't need to call a separate embedding
#### step ourselves.


In [4]:
collection = client.create_collection(name="airline_competitor_intel")

collection.add(
    documents=[doc["text"] for doc in documents],
    ids=[doc["id"] for doc in documents]
)

print(f"Collection created with {collection.count()} documents")

Collection created with 17 documents


#### Test retrieval: does it find the right documents for a real question
#### Asking a question using different wording than the corpus itself uses, to
#### confirm the tool is matching by meaning, not just exact keyword overlap.

In [5]:
query = "Is any competitor cutting back their budget/low-cost service?"

results = collection.query(
    query_texts=[query],
    n_results=3
)

print(f"Query: {query}\n")
for i, (doc_id, doc_text, distance) in enumerate(zip(
    results['ids'][0], results['documents'][0], results['distances'][0]
)):
    print(f"Match {i+1} (distance: {distance:.3f}) [{doc_id}]")
    print(f"  {doc_text}\n")

Query: Is any competitor cutting back their budget/low-cost service?

Match 1 (distance: 1.272) [nk_5]
  Spirit's shutdown effectively removed the primary ultra-low-cost carrier from many US domestic routes, reducing the competitive/price-checking pressure that budget carriers had historically placed on legacy airline fares.

Match 2 (distance: 1.320) [b6_3]
  JetBlue's 'JetForward' turnaround plan emphasizes premium and loyalty revenue growth; transatlantic revenue per available seat mile rose 28% despite reduced capacity, even as core domestic demand softened in early 2025.

Match 3 (distance: 1.340) [sw_5]
  Analysts and industry commentators including Barclays and J.P. Morgan characterized Southwest's pre-2024 commercial strategy as outdated, citing its slower adoption of ancillary revenue streams compared to peers.



#### Generation: have Claude answer using the retrieved documents
#### Takes the top retrieval result and asks Claude to write an actual answer,
#### explicitly grounded in that document, rather than answering from its own
#### general knowledge.

In [6]:
client_anthropic = anthropic.Anthropic()

context = "\n\n".join(results['documents'][0])

prompt = f"""Answer the question using ONLY the information in the context below. If the context doesn't fully answer the question, say what it does cover and note any gaps.

Context:
{context}

Question: {query}

Answer:"""

response = client_anthropic.messages.create(
    model="claude-sonnet-4-5",
    max_tokens=300,
    messages=[{"role": "user", "content": prompt}]
)

print(response.content[0].text)

Based on the context provided:

**What it does cover:**
- Spirit (an ultra-low-cost carrier) had a shutdown that removed it from many US domestic routes
- JetBlue is shifting away from its traditional model, emphasizing premium and loyalty revenue growth rather than budget service, and is reducing domestic capacity while focusing on higher-revenue transatlantic routes

**Gaps:**
The context doesn't explicitly state whether JetBlue is intentionally "cutting back" budget/low-cost service as a strategic decision, though the shift to premium suggests a move away from budget positioning. There's no information about other budget/low-cost carriers' strategies.
